# Figure 1E — HAEC inflammation heatmap

Reproduces the manuscript panel: *"analysis of inflammation markers from bulk
RNA-seq data in HAECs at day 0 and day 40 treated with or without VEGF reveals
that VEGF partially attenuated inflammatory gene expression (Figure 1E)."*

**Input**: `Fig1E_HAEC_FPKM.csv` — Cuffdiff FPKM, 3 samples (HAEC day 0, day 40,
day 40 + VEGF; GEO libraries S10, S11, S14), no biological replicates (n = 1
per condition, as in the original figure).

**Pipeline** (manuscript Methods, "Bulk RNA-seq data analysis"):
1. Load the FPKM matrix (already merged/library-size-and-length normalized by
   Cufflinks).
2. Transform with log10(FPKM + 1).
3. Subset to the curated inflammation/EC-identity gene panel shown in the
   published figure.
4. Row-normalize (min-max per gene) so every gene runs from its own minimum
   (blue) to its own maximum (yellow), matching the lab's standard heatmap
   style.
5. Hierarchically cluster genes (rows); samples (columns) are kept in
   chronological order (D0 -> D40_Ctrl -> D40_VEGF) since there are only 3
   conditions and the biological ordering is the informative one.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

OUT = Path("figures"); OUT.mkdir(exist_ok=True)
INPUT = Path("../Fig1E_HAEC_1rep/processed/Fig1E_HAEC_FPKM.csv")

data = pd.read_csv(INPUT, index_col=0)
print(data.shape, "genes x", data.shape[1] - 1, "samples")
data.head()

In [ ]:
# curated panel, in the order shown in the published figure
PANEL = ["IL6", "SERPINE1", "IL1B", "CXCL8", "APOE", "NFKB1",
         "SOD2", "CCL2", "ICAM1", "VCAM1", "PECAM1", "CD36"]
DISPLAY = {"IL6": "IL6 (Interleukin 6)",
           "SERPINE1": "SERPINE1 (Plasminogen Activator Inhibitor 1)",
           "IL1B": "IL1B (Interleukin 1 Beta)",
           "CXCL8": "CXCL8 (C-X-C Motif Chemokine Ligand 8, also known as IL8)",
           "APOE": "APOE (Apolipoprotein E)",
           "NFKB1": "NFKB1 (Nuclear Factor Kappa B Subunit 1)",
           "SOD2": "SOD2 (Superoxide Dismutase 2)",
           "CCL2": "MCP1 (Monocyte Chemoattractant Protein-1, also known as CCL2)",
           "ICAM1": "ICAM1 (Intercellular Adhesion Molecule 1)",
           "VCAM1": "VCAM1 (Vascular Cell Adhesion Molecule 1)",
           "PECAM1": "PECAM1 (Platelet Endothelial Cell Adhesion Molecule)",
           "CD36": "CD36 (Cluster of Differentiation 36)"}

SAMPLE_ORDER = ["D0", "D40_Ctrl", "D40_VEGF"]
SAMPLE_LABEL = {"D0": "D0", "D40_Ctrl": "D40_Ctrl", "D40_VEGF": "D40_VEGF"}

# a handful of Ensembl IDs did not map to a symbol during matrix build (see
# Fig1E_HAEC_FPKM.csv provenance); patch the known ones by Ensembl ID before
# indexing by symbol, so the curated panel below resolves cleanly
KNOWN_FIX = {"ENSG00000112096": "SOD2"}
fixed = data["gene_official"].copy()
for eid, sym in KNOWN_FIX.items():
    if eid in data.index:
        fixed.loc[eid] = sym
by_gene = data[SAMPLE_ORDER].set_axis(fixed, axis=0)
by_gene = by_gene[~by_gene.index.duplicated(keep="first")]
missing = [g for g in PANEL if g not in by_gene.index]
if missing:
    raise SystemExit(f"genes not found in matrix: {missing}")
panel = by_gene.loc[PANEL]
panel.columns = [SAMPLE_LABEL[c] for c in panel.columns]
panel

In [ ]:
log_fpkm = np.log10(panel + 1)

# row-normalize (min-max per gene) -- "row min" -> "row max" scale, matching
# the lab's house heatmap style used throughout the rest of the manuscript
row_min = log_fpkm.min(axis=1)
row_max = log_fpkm.max(axis=1)
row_range = (row_max - row_min).replace(0, 1)
scaled = log_fpkm.sub(row_min, axis=0).div(row_range, axis=0)

scaled.index = [DISPLAY[g] for g in scaled.index]
scaled.to_csv(OUT / "Fig1E_row_normalized_values.csv")
scaled.round(3)

In [ ]:
cmap = LinearSegmentedColormap.from_list("bky", ["#1600f9", "black", "#e9fa01"], N=256)

# fixed row order matching the published panel (no clustering); samples kept
# in chronological order (D0 -> D40_Ctrl -> D40_VEGF)
g = sns.clustermap(scaled, cmap=cmap, row_cluster=False, col_cluster=False,
                    figsize=(4.5, 6.5), cbar_kws={"label": "row min -> row max"},
                    dendrogram_ratio=0.001, linewidths=0.5, linecolor="white")
g.ax_heatmap.set_ylabel("")
g.ax_heatmap.set_xlabel("")
g.savefig(OUT / "Fig1E_HAEC_inflammation_heatmap.pdf", bbox_inches="tight")
g.savefig(OUT / "Fig1E_HAEC_inflammation_heatmap.png", dpi=300, bbox_inches="tight")
print("wrote", OUT / "Fig1E_HAEC_inflammation_heatmap.png")